In [1]:
import os
import torch

try:
    from google.colab import userdata
    access_token = userdata.get("HF_TOKEN")
except ImportError:
    from dotenv import load_dotenv

    load_dotenv()
    access_token = os.environ.get("HF_TOKEN")

if not access_token:
    raise ValueError(
        "HF_TOKEN was not found. Add it to Colab Secrets or your local .env file."
    )

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

device

device(type='mps')

In [2]:
from transformers import AutoTokenizer

tokenizer_name = "bigcode/starcoder2-3b"
tokenizer = AutoTokenizer.from_pretrained(tokenizer_name, token=access_token)

model_config = {
    "tokenizer_name": tokenizer_name,
    "batch_size": 4,
    "context_length": 1024,
    "vocab_size": len(tokenizer),
    "embedding_dim": 256,
    "n_layers": 2,
    "n_heads": 4,
    "n_kv_heads": 2,
}

batch_size = model_config["batch_size"]
context_length = model_config["context_length"]
vocab_size = model_config["vocab_size"]
embedding_dim = model_config["embedding_dim"]
n_layers = model_config["n_layers"]
n_heads = model_config["n_heads"]
n_kv_heads = model_config["n_kv_heads"]

/Users/kuzeyaldemir/Projects/tiny-coder/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 49151), got 50256. This may result in unexpected behavior.
[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 49151), got 50256. This may result in unexpected behavior.


In [3]:
from data import create_python_dataset, CoderDataset
from torch.utils.data import DataLoader

train_raw = create_python_dataset("data/python-edu/train.jsonl")
val_raw = create_python_dataset("data/python-edu/val.jsonl")

train_set = CoderDataset(train_raw, tokenizer, context_length)
val_set = CoderDataset(val_raw, tokenizer, context_length)

train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=False, num_workers=0)
val_loader = DataLoader(val_set, batch_size=batch_size, shuffle=False, num_workers=0)

x_sample, y_sample = next(iter(train_loader))
print("x shape:", x_sample.shape)
print("y shape:", y_sample.shape)
print("shift check:", torch.equal(x_sample[:, 1:], y_sample[:, :-1]))

x shape: torch.Size([4, 1024])
y shape: torch.Size([4, 1024])
shift check: True


In [4]:
def generate_tokens(input, num_tokens, model):
    encoded = tokenizer.encode(input, return_tensors="pt").to(device)

    model.eval()
    with torch.no_grad():
        for _ in range(num_tokens):
            model_input = encoded[:, -context_length:]
            output = model(model_input)
            next_token_pred = torch.argmax(output[:, -1, :], dim=-1).unsqueeze(1)
            encoded = torch.cat((encoded, next_token_pred), dim=1)
    return encoded

In [5]:
def evaluate_model(data_loader, model, loss_fn):
    model.eval()
    losses = []

    with torch.no_grad():
        for x, y in iter(data_loader):
            x, y = x.to(device), y.to(device)
            logits = model(x)
            loss = loss_fn(torch.flatten(logits, end_dim=1), torch.flatten(y))
            losses.append(loss.item())

    return sum(losses) / len(losses)

In [6]:
def train_model(train_loader, val_loader, model, optimizer, loss_fn, max_steps=1000, eval_interval=20):
    model.train()
    train_losses = []
    val_losses = []

    for step, (x, y) in enumerate(train_loader, start=1):
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()

        logits = model(x)
        loss = loss_fn(torch.flatten(logits, end_dim=1), torch.flatten(y))
        loss.backward()
        optimizer.step()
        train_losses.append(loss.item())

        if step % eval_interval == 0:
            val_loss = evaluate_model(val_loader, model, loss_fn)
            val_losses.append((step, val_loss))
            print(
                f"Step {step}: "
                f"Train loss = {loss.item():.4f} "
                f"Val loss = {val_loss:.4f}"
            )
            model.train()

        if step >= max_steps:
            break

    return train_losses, val_losses, step

In [7]:
from model import LLM
from torch import nn, optim

model = LLM(
    vocab_size,
    context_length,
    embedding_dim,
    n_layers,
    n_heads,
    n_kv_heads,
    intermediate_dim=embedding_dim * 4,
).to(device)

loss_fn = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=3e-4)

In [8]:
initial_val_loss = evaluate_model(val_loader, model, loss_fn)

train_losses, val_losses, completed_steps = train_model(
    train_loader,
    val_loader,
    model,
    optimizer,
    loss_fn,
    max_steps=100,
    eval_interval=20,
)

final_val_loss = evaluate_model(val_loader, model, loss_fn)
decoded = tokenizer.decode(generate_tokens("def hello_world", 10, model)[0])

print("Initial val loss:", initial_val_loss)
print("Final val loss:", final_val_loss)
print("Completed steps:", completed_steps)
print("Generated text:", decoded)

Step 20: Train loss = 9.7855 Val loss = 9.5922
Step 40: Train loss = 7.4088 Val loss = 7.5036
Step 60: Train loss = 7.4615 Val loss = 6.9543
Step 80: Train loss = 6.8419 Val loss = 6.8855
Step 100: Train loss = 6.7422 Val loss = 6.8597


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Initial val loss: 10.851842744009835
Final val loss: 6.859673976898193
Completed steps: 100
Generated text: def hello_world












In [ ]:
from pathlib import Path

Path("checkpoints").mkdir(exist_ok=True)
checkpoint_path = f"checkpoints/base_model_{completed_steps}steps.pt"

torch.save(
    {
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "step": completed_steps,
        "model_config": model_config,
        "final_val_loss": final_val_loss,
    },
    checkpoint_path,
)

checkpoint_path

In [ ]:
checkpoint = torch.load(checkpoint_path, map_location=device)

model.load_state_dict(checkpoint["model_state_dict"])
optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
completed_steps = checkpoint["step"]

completed_steps